In [1]:
import os
import pickle

import pandas as pd
from tqdm.auto import tqdm

import autoslo.utils.paths as pu
from autoslo.forecasting.arrival_classifier import ArrivalClassifier
from autoslo.workload_definition.query import Query

import pyarrow as pa
import multiprocessing as mp

cluster_type = "provisioned"

columns = [
    "database_id",
    "query_id",
    "feature_fingerprint",
    "query_type",
    "num_permanent_tables_accessed",
    "num_external_tables_accessed",
    "num_system_tables_accessed",
    "read_table_ids",
    "write_table_ids",
    "arrival_timestamp",
]

In [2]:
cluster_id = 12
path = pu.get_redset_raw_data(
    cluster_type=cluster_type, cluster_id=cluster_id
)
out_dir = os.path.join(
    pu.get_data_path(),
    "redset_byproducts",
    cluster_type,
    str(cluster_id),
)
os.makedirs(out_dir, exist_ok=True)
workload_df_path = os.path.join(out_dir, f"workload.parquet")


In [3]:
df = pd.read_parquet(path, columns=columns)

df["template_str"] = df.apply(
    lambda row: "_".join(
        [
            str(row["database_id"]),
            str(row["feature_fingerprint"]),
            str(row["query_type"]),
            # str(row["num_permanent_tables_accessed"]),
            # str(row["num_external_tables_accessed"]),
            # str(row["num_system_tables_accessed"]),
            # str(row["read_table_ids"]),
            # str(row["write_table_ids"]),
        ]
    ),
    axis=1,
)

In [5]:
mapping = {}
for i, template_str in enumerate(sorted(df["template_str"].unique())):
    mapping[template_str] = i
df["query_template"] = df["template_str"].map(mapping)

mapping_path = os.path.join(out_dir, f"template_mapping.pkl")
with open(mapping_path, "wb") as f:
    pickle.dump(mapping, f)

In [6]:
queries = []
for i, row in df.iterrows():
    queries.append(
        Query(
            query_id=row["query_id"],
            tpcds_temp_and_q_idx=f"{row['query_template']}",
            abs_start_time=row["arrival_timestamp"]
        )
    )
queries_path = os.path.join(out_dir, f"queries.pkl")

with open(queries_path, "wb") as f:
    pickle.dump(queries, f)

In [9]:
df['query_template'].value_counts()

query_template
221572    509685
755756    286846
755752     94993
221571     86283
755753     69443
           ...  
602938         1
845345         1
110264         1
818817         1
435547         1
Name: count, Length: 1210757, dtype: int64

In [15]:
df[df['query_template'] == 755753].head()

,database_id,query_id,feature_fingerprint,query_type,num_permanent_tables_accessed,num_external_tables_accessed,num_system_tables_accessed,read_table_ids,write_table_ids,arrival_timestamp,template_str,query_template
59,0,4259154,None,ctas,26.0,0.0,0.0,"104,106,107,108,47,57",783713,2024-03-01 00:01:30.916023,0_None_ctas,755753
163,0,3884249,None,ctas,6.0,0.0,0.0,"134,31,32,33,34,35",693365,2024-03-01 00:04:01.622062,0_None_ctas,755753
235,0,4177802,None,ctas,26.0,0.0,0.0,"104,106,107,108,47,57",771450,2024-03-01 00:07:50.329505,0_None_ctas,755753
410,0,3650770,None,ctas,26.0,0.0,0.0,"104,106,107,108,47,57",672965,2024-03-01 00:11:56.379406,0_None_ctas,755753
556,0,3331071,None,ctas,26.0,0.0,0.0,"104,106,107,108,47,57",629151,2024-03-01 00:15:21.611043,0_None_ctas,755753


In [18]:
classification_path = os.path.join(out_dir, f"template_classification.pkl")

classifier = ArrivalClassifier(queries=queries, verbose=False)
classifier.classify_arrivals()

with open(classification_path, "wb") as f:
    pickle.dump(
        {
            "classification": classifier._template_classification,
            "details": classifier._template_details,
        },
        f,
    )

/home/markakis/markos-3.11.9-venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/markakis/markos-3.11.9-venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
